# Customer Churn Analysis Using Bayesian Thinking

This notebook explores customer churn using probability and conditional probability,
with a focus on **Bayes’ theorem as belief updating**, rather than mathematical formulas
or predictive modeling.

The goal is to understand how observing customer attributes changes our belief
about whether a customer is likely to churn.

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/telco-customer-churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Q1. Overall Churn Probability (Prior Belief)

What is the overall probability that a customer churns?

This represents the **prior belief** about churn before observing
any customer-specific information.

In [5]:
# Overall probability of churn (prior)
prior_churn_probability = (df["Churn"] == "Yes").mean()
prior_churn_probability

np.float64(0.2653698707936959)

### Interpretation

The overall probability of churn is approximately **26.5%**.

This means that, before observing any customer-specific information,
about **1 in 4 customers** in this dataset eventually churn.

This value represents the **prior belief** in a Bayesian sense.
All subsequent conditional probabilities will be interpreted as
updates to this baseline belief after observing additional evidence
about a customer.

### Q2. Churn Probability Given Contract Type

How does the probability of churn change based on the type of contract
a customer is on?

Contract type is treated as **evidence** that updates our prior belief
about customer churn.


In [6]:
# Churn probability conditioned on contract type
churn_probability_by_contract = (
    df.groupby("Contract")["Churn"]
      .apply(lambda x: (x == "Yes").mean())
      .sort_values(ascending=False)
)

churn_probability_by_contract

Contract
Month-to-month    0.427097
One year          0.112695
Two year          0.028319
Name: Churn, dtype: float64

### Interpretation

Contract type has a **strong influence** on churn probability.

- Customers on **month-to-month contracts** have a churn probability of approximately **42.7%**,
  which is significantly higher than the overall baseline of 26.5%.
- Customers on a **one-year contract** have a much lower churn probability of about **11.3%**.
- Customers on a **two-year contract** show the lowest churn probability at roughly **2.8%**.

From a Bayesian perspective, observing that a customer is on a month-to-month contract
substantially increases our belief that they may churn, while longer-term contracts
strongly decrease that belief.

### Q3. Churn Probability Given Tech Support

How does the probability of churn change based on whether a customer
has access to technical support?

Tech support is treated as **evidence** that may update our belief
about a customer's likelihood of churn.

In [7]:
# Churn probability conditioned on Tech Support availability
churn_probability_by_techsupport = (
    df.groupby("TechSupport")["Churn"]
      .apply(lambda x: (x == "Yes").mean())
      .sort_values(ascending=False)
)

churn_probability_by_techsupport

TechSupport
No                     0.416355
Yes                    0.151663
No internet service    0.074050
Name: Churn, dtype: float64

### Interpretation

Technical support availability has a strong relationship with customer churn.

- Customers with **no tech support** have a churn probability of approximately **41.6%**,
  which is significantly higher than the overall baseline of 26.5%.
- Customers **with tech support** churn at a much lower rate of about **15.2%**.
- Customers with **no internet service** have the lowest churn probability (**~7.4%**),
  likely because they are subscribed to different, more stable service offerings.

From a Bayesian perspective, observing that a customer lacks tech support
substantially increases our belief that they may churn, while the presence
of tech support decreases that belief.

### Q4. Churn Probability Given Contract Type and Tech Support

How does the probability of churn change when **multiple pieces of evidence**
are observed together?

In particular, how does churn risk change for customers who are:
- on a month-to-month contract, and
- do not have technical support?

This reflects a Bayesian belief update using combined evidence.


In [8]:
# Filter customers on month-to-month contracts
month_to_month = df[df["Contract"] == "Month-to-month"]

# Churn probability by Tech Support within month-to-month customers
churn_prob_contract_and_support = (
    month_to_month.groupby("TechSupport")["Churn"]
                  .apply(lambda x: (x == "Yes").mean())
                  .sort_values(ascending=False)
)

churn_prob_contract_and_support

TechSupport
No                     0.503731
Yes                    0.307004
No internet service    0.188931
Name: Churn, dtype: float64

### Interpretation

When combining contract type and technical support, churn risk increases further,
demonstrating the effect of **multiple pieces of evidence** on belief updating.

- Month-to-month customers **without tech support** have a churn probability of approximately **50.4%**,
  which is substantially higher than both the overall baseline (26.5%) and the individual signals alone.
- Month-to-month customers **with tech support** still exhibit elevated churn risk (**~30.7%**),
  but significantly lower than those without support.
- Month-to-month customers with **no internet service** show lower churn probability (**~18.9%**),
  consistent with earlier observations.

From a Bayesian perspective, observing *both* a month-to-month contract and lack of tech support
leads to a much stronger belief that a customer is likely to churn than observing either factor in isolation.

### Q5. Which Factor Provides the Strongest Signal of Churn?

Among contract type, technical support, and their combination,
which factor most strongly increases the probability of customer churn?

This question synthesizes all previous analysis to identify
the most influential churn risk signal.

In [9]:
# Prior churn probability
prior = prior_churn_probability

signal_strength_churn = {
    "Month-to-month contract": churn_probability_by_contract.loc["Month-to-month"] / prior,
    "No Tech Support": churn_probability_by_techsupport.loc["No"] / prior,
    "Month-to-month + No Tech Support": churn_prob_contract_and_support.loc["No"] / prior
}

signal_strength_churn

{'Month-to-month contract': np.float64(1.609439583009717),
 'No Tech Support': np.float64(1.5689600906603982),
 'Month-to-month + No Tech Support': np.float64(1.8982235691526312)}

### Final Interpretation: Strongest Signal of Customer Churn

Among the factors analyzed, the **combination of a month-to-month contract and lack of technical support**
provides the strongest signal of customer churn.

- Being on a **month-to-month contract** increases churn probability by approximately **1.61×**
  compared to the overall baseline.
- Having **no tech support** increases churn probability by about **1.57×**.
- Observing **both conditions together** increases churn probability by nearly **1.90×**,
  representing the strongest shift from the prior belief.

This demonstrates a core Bayesian insight: **multiple independent risk signals compound to produce
a stronger belief update than any single signal alone**.